<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_07_2_learnable_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 7: PyTorch Building Blocks**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 7 Material


* Part 7.1: Model Structure [[Video]]() [[Notebook]](t81_558_class_07_1_model_structure.ipynb)
* **Part 7.2: Learnable Layers** [[Video]]() [[Notebook]](t81_558_class_07_2_learnable_layers.ipynb)
* Part 7.3: Nonlinearities (Activations) [[Video]]() [[Notebook]](t81_558_class_07_3_transfer.ipynb)
* Part 7.4: Normalization & Regularization [[Video]]() [[Notebook]](t81_558_class_07_4_normalization.ipynb)
* Part 7.5: Shape [[Video]]() [[Notebook]](t81_558_class_07_5_shapes.ipynb)



# Google CoLab Instructions

The following code checks that Google CoLab is and sets up the correct hardware settings for PyTorch.


In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cpu


# Part 7.2: Learnable Layers

Learnable layers form the core of a neural network. Each layer applies a transformation to the input using parameters that are adjusted during training. Examples include fully connected layers, convolutional layers, recurrent layers, and embeddings. These layers define how information flows and how patterns are represented.

## Fully Connected Layers

The fully connected layer, often called a dense layer or linear layer, is the most general learnable layer. Every input value contributes to every output value through a learned weight, and each output also has its own learned bias. In PyTorch this layer is implemented as `nn.Linear`. Mathematically, it computes the affine transformation y = xW^T + b, where W is a weight matrix of shape (out_features, in_features) and b is a bias vector of length out_features.

Fully connected layers are appropriate when there is no special structure in the input that should be preserved. They are the workhorse of tabular models and are also used as the final classification or regression head on top of convolutional or recurrent backbones. Their main weakness is parameter count. A layer that maps 1,000 inputs to 1,000 outputs already contains roughly one million weights, which grows quickly and can lead to overfitting on smaller datasets.

## Convolutional Layers

Convolutional layers exploit spatial structure. Instead of connecting every input to every output, a convolutional layer slides a small set of learnable filters across the input and produces a feature map that records how strongly each filter responds at each location. Because the same filter is reused at every position, the layer has far fewer parameters than a fully connected layer of comparable capacity, and it learns features that are translation equivariant.

PyTorch provides `nn.Conv1d`, `nn.Conv2d`, and `nn.Conv3d` for one, two, and three dimensional inputs. The two dimensional version is the most common and is the foundation of nearly every modern image model. Key hyperparameters include the number of input channels, the number of output channels (the number of filters), the kernel size, the stride, and the padding. Together these control how the spatial dimensions of the input are transformed into the spatial dimensions of the output.

## Recurrent Layers

Recurrent layers are designed for sequential data such as text, audio, or time series. They process one element of the sequence at a time and maintain a hidden state that carries information from previous steps forward. The same set of weights is applied at every time step, which lets the layer handle sequences of arbitrary length with a fixed number of parameters.

PyTorch offers three main recurrent layers. `nn.RNN` is the simplest variant and uses a single tanh or ReLU update. `nn.LSTM` adds gating mechanisms and a separate cell state, which helps the network preserve information over longer spans and reduces the vanishing gradient problem. `nn.GRU` is a streamlined alternative to LSTM with fewer parameters and similar performance on many tasks. For most modern sequence work, transformer based attention layers have largely replaced recurrent layers, but RNN family layers remain useful for streaming data, low latency inference, and smaller models.

## Embedding Layers

Embedding layers turn discrete tokens into dense continuous vectors. They are essentially a lookup table where each row is a learned vector that represents one item in a vocabulary. When the model receives an integer index, the embedding layer returns the corresponding row. During training, the rows are updated so that semantically related tokens end up with similar vectors.

In PyTorch, `nn.Embedding` takes the size of the vocabulary and the dimension of the embedding vectors as its main arguments. Embeddings are most familiar from natural language processing, where each word or subword token gets its own vector, but the same idea applies to any categorical input. User identifiers in a recommendation system, product categories in a sales model, and even discrete state labels in reinforcement learning are all natural fits for embedding layers.

## Attention and Transformer Layers

Attention layers learn to weight different parts of the input based on their relevance to each output position. The core operation is scaled dot product attention, which compares a set of queries against a set of keys and uses the resulting scores to produce a weighted average of values. Multi head attention runs several attention operations in parallel with different learned projections, which lets the layer attend to several patterns at once.

PyTorch provides `nn.MultiheadAttention` for the raw mechanism and `nn.TransformerEncoderLayer`, `nn.TransformerDecoderLayer`, and the higher level `nn.Transformer` for full transformer blocks. These layers have become the default choice for language modeling and are increasingly used in vision, audio, and scientific applications as well.

## How Parameters Are Learned

Every learnable layer in PyTorch registers its parameters as `nn.Parameter` objects, which the framework tracks automatically. When you call `model.parameters()`, PyTorch returns a flat iterator over every weight and bias in every layer, which is exactly what an optimizer such as `torch.optim.Adam` expects.

During training, a forward pass computes predictions and a loss. A backward pass uses automatic differentiation to compute the gradient of the loss with respect to every parameter. The optimizer then nudges each parameter in the direction that reduces the loss. Because all learnable layers follow this same protocol, you can mix linear, convolutional, recurrent, embedding, and attention layers freely inside a single `nn.Module` and rely on the same training loop for all of them.

## Choosing the Right Layer

The choice of layer should follow the structure of the data. Tabular features with no inherent ordering are well served by fully connected layers. Grid structured data such as images or spectrograms benefits from convolutional layers because nearby pixels are related and the same patterns can appear anywhere in the frame. Sequences with strong order dependence can use recurrent layers when memory and latency matter, or transformer layers when accuracy and parallelism matter more. Categorical inputs with large vocabularies almost always pass through embedding layers before reaching the rest of the network.

Most real models combine several of these layer types. A text classifier might begin with an embedding layer, pass the result through a transformer encoder, and finish with a fully connected head. An image captioning model might use convolutional layers for the image, embeddings for the tokens, and attention layers to bridge the two. Understanding what each layer offers makes it easier to design architectures that match the problem at hand.

## Inspecting Learnable Layers in PyTorch

The following example builds a small model that uses several of the learnable layer types discussed above. It then walks through the model and prints the shape and parameter count of each layer. This is a useful pattern when debugging an architecture or reporting model size.

In [2]:
import torch
from torch import nn

class MixedLayerModel(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=64, num_classes=10):
        super().__init__()

        # Embedding layer for token ids
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embed_dim)

        # 1D convolution over the embedded sequence
        # Input shape after permute: (batch, embed_dim, seq_len)
        self.conv = nn.Conv1d(in_channels=embed_dim,
                              out_channels=128,
                              kernel_size=3,
                              padding=1)

        # GRU recurrent layer
        self.gru = nn.GRU(input_size=128,
                          hidden_size=64,
                          batch_first=True)

        # Fully connected classification head
        self.fc1 = nn.Linear(in_features=64, out_features=32)
        self.fc2 = nn.Linear(in_features=32, out_features=num_classes)

    def forward(self, token_ids):
        # token_ids shape: (batch, seq_len)
        x = self.embedding(token_ids)            # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)                   # (batch, embed_dim, seq_len)
        x = torch.relu(self.conv(x))             # (batch, 128, seq_len)
        x = x.permute(0, 2, 1)                   # (batch, seq_len, 128)
        _, h = self.gru(x)                       # h: (1, batch, 64)
        h = h.squeeze(0)                         # (batch, 64)
        h = torch.relu(self.fc1(h))              # (batch, 32)
        return self.fc2(h)                       # (batch, num_classes)


model = MixedLayerModel()

# Walk through every named layer and report its parameter count.
print(f"{'Layer':<20}{'Type':<25}{'Parameters':>12}")
print("-" * 57)

total_params = 0
for name, module in model.named_children():
    layer_params = sum(p.numel() for p in module.parameters())
    total_params += layer_params
    print(f"{name:<20}{type(module).__name__:<25}{layer_params:>12,}")

print("-" * 57)
print(f"{'Total':<45}{total_params:>12,}")

# Run a quick forward pass to confirm shapes line up.
sample = torch.randint(low=0, high=10000, size=(4, 20))  # batch of 4, seq len 20
output = model(sample)
print(f"\nInput shape:  {tuple(sample.shape)}")
print(f"Output shape: {tuple(output.shape)}")

Layer               Type                       Parameters
---------------------------------------------------------
embedding           Embedding                     640,000
conv                Conv1d                         24,704
gru                 GRU                            37,248
fc1                 Linear                          2,080
fc2                 Linear                            330
---------------------------------------------------------
Total                                             704,362

Input shape:  (4, 20)
Output shape: (4, 10)
